# Processing Drought Data

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path

# Define Paths relative to project root
PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / 'Data' / 'Raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'Processed'

# Load the data
coords_df = pd.read_csv(RAW / 'CSV/drought_coords.csv')
values_df = pd.read_csv(RAW / 'CSV/drought_values.csv')

# Handle missing values
values_df = values_df.replace(-99.999, np.nan)

# Define the target year
target_year = 1536

# Windows: 1, 2, 3, 5, 10 years leading up to and including 1536
windows = {
    '1yr': [1536],
    '2yr': [1535, 1536],
    '3yr': [1534, 1535, 1536],
    '5yr': list(range(1532, 1537)),
    '10yr': list(range(1527, 1537))
}

results = {}

# Single-year wet weather (1535)
year_1535_df = values_df[values_df['year'] == 1535]
if not year_1535_df.empty:
    results['pdsi_1535'] = year_1535_df.drop(columns=['year']).mean()

# Single-year wet weather (1536)
year_1536_df = values_df[values_df['year'] == 1536]
if not year_1536_df.empty:
    results['pdsi_1536'] = year_1536_df.drop(columns=['year']).mean()

if 'pdsi_1535' in results and 'pdsi_1536' in results:
    results['pdsi_1535x1536'] = results['pdsi_1535'] * results['pdsi_1536']

for name, years in windows.items():
    window_df = values_df[values_df['year'].isin(years)]
    means = window_df.drop(columns=['year']).mean()
    means = -1 * means  # Flip so higher values indicate more severe drought
    abs_means = window_df.drop(columns=['year']).abs().mean()
    results[f'pdsi_avg_{name}'] = means
    results[f'pdsi_ext_{name}'] = abs_means

# Combine results into a single DataFrame
results_df = pd.DataFrame(results)
results_df.index = results_df.index.astype(int)
results_df.index.name = 'Grid-cell'

# Merge with coordinates
merged_df = coords_df.merge(results_df, on='Grid-cell', how='inner')

# Convert to GeoDataFrame (coords in WGS84)
geometry = [Point(xy) for xy in zip(merged_df['Longitude'], merged_df['Latitude'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

# Reproject to British National Grid (EPSG:27700)
gdf_bng = gdf.to_crs("EPSG:27700")
gdf_bng['bng_x'] = gdf_bng.geometry.x
gdf_bng['bng_y'] = gdf_bng.geometry.y

# Drop the geometry column for CSV output
output_path = PROCESSED / 'drought_intensity_bng.csv'
pd.DataFrame(gdf_bng.drop(columns='geometry')).to_csv(output_path, index=False)
print(f"Processed drought data saved to {output_path}")

Processed drought data saved to c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\drought_intensity_bng.csv
